# Recherche parallèle : les règles du contentieux sous la codification anticipée

*Recherche parallèle, descriptive, **sans modèle**. Journal : [`tableau_resultats.md`](tableau_resultats.md).*

**Codification anticipée** (convention B des notebooks `05` et suivants) : la codification `PAY_n` du mois M porte sur la facture du mois M (`BILL_AMTn`), réglée le mois suivant par `PAY_AMT(n-1)`. **Convention actuelle** (A, celle du projet) : `PAY_n` porte sur `BILL_AMT(n+1)`, réglée par `PAY_AMTn`.

Trois règles du contentieux (niveau 5 du nettoyage, définition 2, `src/02_01_nettoyage.ipynb`) reposent sur la convention actuelle. **Question** : les incohérences qu'elles corrigent existent-elles encore sous la codification anticipée, et leurs présomptions tiennent-elles ?
1. **Faux retards neutralisés** : une entrée en retard (`PAY_m` >= 2 alors que `PAY_(m+1)` < 2) posée sur une facture nulle, `BILL_AMT(m+1)` <= 0 ; la série de retards est remplacée par la codification saine d'avant.
2. **Retard payé** : un retard en septembre (`PAY_1` >= 2) dont la facture est payée à 90 % ou plus en septembre (`PAY_AMT1` / `BILL_AMT2`) ou en août (`PAY_AMT2` / `BILL_AMT3`) est présumé sorti en octobre : il n'est pas compté au contentieux à M.
3. **Mois de transition recodé** : un client en retard en août qui paraît sortir en septembre (`PAY_1` <= 1) est remis à **2** s'il n'a rien payé en août ni en septembre alors que deux factures étaient dues ; sa sortie est confirmée s'il a payé au moins 90 % de sa facture de septembre.

Les règles 2 et 3 portent sur septembre, où la codification anticipée ne se calcule pas (paiement d'octobre inconnu). Elles sont donc **testées sur les mois d'avant**, où la suite est connue : le même raisonnement, appliqué à un mois M d'avril à août, est-il confirmé par la codification du mois suivant ?

**Données** : niveau 0, sans `dpnm`, sans les clients du test du ML de septembre (comme le `05`).

## 1. Données

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv").drop(columns="dpnm")   # dpnm jamais utilisé
connus = df[df["PAY_1"] != 1]
_, test_ml = train_test_split(connus.index, test_size=0.2, stratify=connus["PAY_1"], random_state=42)
df = df.drop(index=test_ml)
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}
print(f"Clients retenus : {len(df)}")

def ratio(paiement, facture):
    # Part de la facture payée, en % ; 0 si aucune facture (pas de preuve de paiement), comme dans le niveau 5
    return (paiement / facture.where(facture > 0) * 100).fillna(0)

Clients retenus : 24737


## 2. Règle 1 : les faux retards (entrée en retard sur une facture nulle)

Pour chaque mois M de septembre à mai, les entrées en retard (`PAY_m` >= 2, `PAY_(m+1)` < 2), et parmi elles celles posées sur une facture nulle :
- **convention A** : facture du mois précédent nulle (`BILL_AMT(m+1)` <= 0), la règle actuelle ;
- **convention B** : facture du mois M nulle (`BILL_AMTm` <= 0).

Pour les faux retards sous A, on regarde aussi ce que dit la codification anticipée : la facture du mois M est-elle positive (le client a dépensé : il y a bien une facture à régler) ?

In [2]:
lignes = []
faux_A_tous, faux_B_tous = [], []
for m in range(1, 6):
    entree = (df[f"PAY_{m}"] >= 2) & (df[f"PAY_{m + 1}"] < 2) & (df[f"PAY_{m + 1}"] != 1)
    faux_A = entree & (df[f"BILL_AMT{m + 1}"] <= 0)
    faux_B = entree & (df[f"BILL_AMT{m}"] <= 0)
    faux_A_tous.append(faux_A.rename(m)); faux_B_tous.append(faux_B.rename(m))
    lignes.append({"Mois M": MOIS[m], "Entrées en retard": int(entree.sum()),
                   "Facture nulle, A (faux retards actuels)": int(faux_A.sum()),
                   "dont facture du mois M positive (B : vraie facture à régler)": int((faux_A & (df[f"BILL_AMT{m}"] > 0)).sum()),
                   "Facture nulle, B": int(faux_B.sum()),
                   "Facture nulle sous A et sous B": int((faux_A & faux_B).sum())})
t = pd.DataFrame(lignes).set_index("Mois M")
t.loc["Total"] = t.sum()
display(t)
FAUX_A = pd.concat(faux_A_tous, axis=1).any(axis=1)
FAUX_B = pd.concat(faux_B_tous, axis=1).any(axis=1)
print(f"Clients avec au moins un faux retard : A {int(FAUX_A.sum())} ; B {int(FAUX_B.sum())} ; les deux {int((FAUX_A & FAUX_B).sum())}")

,Entrées en retard,"Facture nulle, A (faux retards actuels)",dont facture du mois M positive (B : vraie facture à régler),"Facture nulle, B",Facture nulle sous A et sous B
Mois M,,,,,
septembre,799,0,0,0,0
août,1355,38,1,47,37
juillet,1376,69,2,111,67
juin,1073,54,1,85,53
mai,741,34,0,48,34
Total,5344,195,4,291,191


Clients avec au moins un faux retard : A 195 ; B 291 ; les deux 191


**Comment lire** : une ligne par mois M d'entrée en retard. La colonne « dont facture du mois M positive » compte les faux retards actuels qui, sous la codification anticipée, portent en fait sur une vraie facture (le client a dépensé dans le mois) : ce ne sont plus des faux retards. Les faux retards sous B sont ceux qui restent inexpliqués.

## 3. Règle 2 : un retard « payé » sort-il vraiment du retard le mois suivant ?

Pour chaque mois M d'août à mai, les clients en retard en M (`PAY_n` >= 2) ; la codification du mois suivant (`PAY_(n-1)`) dit s'ils sont sortis. On compare la part de sorties :
- **retard « payé » au sens de la règle actuelle (A)** : facture payée à 90 % ou plus le mois M (`PAY_AMTn` / `BILL_AMT(n+1)`) ou le mois d'avant (`PAY_AMT(n+1)` / `BILL_AMT(n+2)`) ;
- **retard « payé » au sens de la codification anticipée (B)** : facture du mois M payée à 90 % ou plus le mois suivant (`PAY_AMT(n-1)` / `BILL_AMTn`) ;
- **tous les retards** (référence).

Si la présomption « retard payé = sortie » est juste, la part de sorties doit être très haute pour les retards « payés ».

In [3]:
lignes = []
for n in range(2, 6):
    r = df[df[f"PAY_{n}"] >= 2]
    r = r[r[f"PAY_{n - 1}"] != 1]
    sortie = r[f"PAY_{n - 1}"] <= 0
    paye_A = (ratio(r[f"PAY_AMT{n}"], r[f"BILL_AMT{n + 1}"]) >= 90) | (ratio(r[f"PAY_AMT{n + 1}"], r[f"BILL_AMT{n + 2}"]) >= 90 if n + 2 <= 6 else False)
    paye_B = ratio(r[f"PAY_AMT{n - 1}"], r[f"BILL_AMT{n}"]) >= 90
    lignes.append({"Mois M": MOIS[n], "Retards": len(r), "Sortent le mois suivant, tous (%)": round(sortie.mean() * 100, 1),
                   "« Payés » A": int(paye_A.sum()), "dont sortent (%)": round(sortie[paye_A].mean() * 100, 1),
                   "« Payés » B": int(paye_B.sum()), "dont sortent (% )": round(sortie[paye_B].mean() * 100, 1)})
display(pd.DataFrame(lignes).set_index("Mois M"))

,Retards,"Sortent le mois suivant, tous (%)",« Payés » A,dont sortent (%),« Payés » B,dont sortent (% )
Mois M,,,,,,
août,2071,17.7,293,81.2,10,40.0
juillet,3597,29.4,428,64.0,17,35.3
juin,2985,25.5,247,53.0,15,20.0
mai,2516,24.0,100,62.0,7,0.0


**Comment lire** : une ligne par mois M. Pour chaque convention, le nombre de retards « payés » et la part qui sort réellement du retard le mois suivant. La règle actuelle du contentieux applique la présomption A à septembre : si elle se vérifie mal les mois d'avant, des clients présumés sortis en octobre sont peut-être encore au contentieux.

## 4. Règle 3 : le mois de transition (sortie apparente d'un retard)

Pour chaque mois M d'août à mai, les clients en retard le mois précédent (`PAY_(n+1)` >= 2) et sains en M (`PAY_n` <= 0) : de vraies sorties, puisque la banque les a codifiées saines et que la codification du mois suivant est connue. On applique la règle 3 à ces sorties :
- **remise à 2 au sens de la règle actuelle (A)** : rien payé le mois M ni le mois d'avant (`PAY_AMTn` = `PAY_AMT(n+1)` = 0) avec deux factures dues (`BILL_AMT(n+1)` et `BILL_AMT(n+2)` > 0) ;
- **remise à 2 au sens de la codification anticipée (B)** : rien payé le mois suivant ni le mois M (`PAY_AMT(n-1)` = `PAY_AMTn` = 0) avec deux factures dues (`BILL_AMTn` et `BILL_AMT(n+1)` > 0) ;
- **sortie confirmée** (90 % ou plus payés) : sous A, `PAY_AMTn` / `BILL_AMT(n+1)` ; sous B, `PAY_AMT(n-1)` / `BILL_AMTn`.

On regarde aussi si ces sorties restent saines le mois d'après (`PAY_(n-1)`), pour juger la règle de remise à 2.

In [4]:
lignes = []
for n in range(2, 6):
    s = df[(df[f"PAY_{n + 1}"] >= 2) & (df[f"PAY_{n}"] <= 0)]
    remise_A = (s[f"PAY_AMT{n}"] == 0) & (s[f"PAY_AMT{n + 1}"] == 0) & (s[f"BILL_AMT{n + 1}"] > 0) & (s[f"BILL_AMT{n + 2}"] > 0 if n + 2 <= 6 else True)
    remise_B = (s[f"PAY_AMT{n - 1}"] == 0) & (s[f"PAY_AMT{n}"] == 0) & (s[f"BILL_AMT{n}"] > 0) & (s[f"BILL_AMT{n + 1}"] > 0)
    confirme_A = ratio(s[f"PAY_AMT{n}"], s[f"BILL_AMT{n + 1}"]) >= 90
    confirme_B = ratio(s[f"PAY_AMT{n - 1}"], s[f"BILL_AMT{n}"]) >= 90
    reste_sain = s[f"PAY_{n - 1}"] <= 0
    lignes.append({"Mois M": MOIS[n], "Sorties": len(s),
                   "Remises à 2, A": int(remise_A.sum()), "dont restent saines le mois d'après (%)": round(reste_sain[remise_A].mean() * 100, 1) if remise_A.any() else np.nan,
                   "Remises à 2, B": int(remise_B.sum()),
                   "Sortie confirmée, A (%)": round(confirme_A.mean() * 100, 1), "Sortie confirmée, B (%)": round(confirme_B.mean() * 100, 1),
                   "Restent saines le mois d'après, toutes (%)": round(reste_sain.mean() * 100, 1)})
display(pd.DataFrame(lignes).set_index("Mois M"))

,Sorties,"Remises à 2, A",dont restent saines le mois d'après (%),"Remises à 2, B","Sortie confirmée, A (%)","Sortie confirmée, B (%)","Restent saines le mois d'après, toutes (%)"
Mois M,,,,,,,
août,1057,84,79.8,1,0.6,28.8,91.0
juillet,762,50,90.0,2,0.4,23.0,96.1
juin,603,16,75.0,0,0.0,15.9,96.7
mai,818,55,90.9,0,0.1,18.0,98.0


**Comment lire** : une ligne par mois M ; toutes ces sorties sont réelles (codifiées saines par la banque). Une règle de remise à 2 qui se déclenche souvent sur de vraies sorties se trompe ; une sortie confirmée par le paiement dans une large majorité des cas valide la convention utilisée.

## 5. Des clients à regarder

Quelques faux retards de la règle 1 (actuels, sous A) et quelques retards « payés » sous A qui ne sortent pas du retard le mois suivant, tirés au hasard (`random_state=42`), affichés en entier ; colonnes groupées selon la codification anticipée : pour chaque mois, de septembre à avril, la codification `PAY_n`, la facture du mois `BILL_AMTn` et le paiement du mois suivant `PAY_AMT(n-1)` (septembre : paiement d'octobre inconnu).

In [5]:
COLONNES_B = ["ID", "LIMIT_BAL"]
for n in range(1, 7):
    COLONNES_B += [f"PAY_{n}", f"BILL_AMT{n}"] + ([f"PAY_AMT{n - 1}"] if n >= 2 else [])
COLONNES_B.append("PAY_AMT6")

def afficher(titre, masque, combien=6):
    if not masque.any():
        return
    display(Markdown(f"**{titre}** ({int(masque.sum())} clients)"))
    display(df.loc[masque[masque].sample(min(combien, int(masque.sum())), random_state=42).index, COLONNES_B].set_index("ID"))

afficher("Faux retards actuels (règle 1, convention A)", FAUX_A)
afficher("Faux retards qui restent sous la codification anticipée (convention B)", FAUX_B)
non_sortis = pd.Series(False, index=df.index)
for n in range(2, 6):
    paye_A = (df[f"PAY_{n}"] >= 2) & ((ratio(df[f"PAY_AMT{n}"], df[f"BILL_AMT{n + 1}"]) >= 90) | (ratio(df[f"PAY_AMT{n + 1}"], df[f"BILL_AMT{n + 2}"]) >= 90 if n + 2 <= 6 else False))
    non_sortis |= paye_A & (df[f"PAY_{n - 1}"] >= 2)
afficher("Retards « payés » sous A qui restent en retard le mois suivant (règle 2)", non_sortis)

**Faux retards actuels (règle 1, convention A)** (195 clients)

,LIMIT_BAL,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,
22287,20000,1,0,-2,0,0,-1,780,780,2,780,0,2,0,0,-2,0,0,0
2712,360000,-1,1605,2,1448,14,2,0,0,-2,0,0,-2,0,0,-2,0,0,0
24667,430000,2,435,2,-183,183,-2,-525,0,-2,-868,0,-2,-1211,0,-2,-1840,290,2512
16383,120000,-1,536,2,210,0,2,-116,0,-2,-442,0,-1,732,1500,0,406,0,0
13378,80000,1,0,-1,1684,1684,2,1509,0,2,-357,0,-2,-357,0,-2,-357,0,0
24458,120000,-1,66,-1,584,843,2,259,0,2,-66,66,-2,-391,325,-2,-716,325,325


**Faux retards qui restent sous la codification anticipée (convention B)** (291 clients)

,LIMIT_BAL,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,
10360,40000,0,12745,0,6889,1500,0,8530,3005,2,4610,0,2,-220,0,-1,1060,2000,4000
27179,120000,2,61650,2,0,0,-2,0,0,-2,0,0,-2,0,0,-2,0,0,0
6371,120000,3,113722,3,110525,0,2,103632,0,2,-36,0,-2,-18,0,-2,-18,0,0
20280,280000,0,5380,-1,390,400,-1,1843,1843,2,1663,0,2,-147,0,-2,-147,0,1647
16039,20000,-1,2120,2,1760,0,2,0,0,-2,0,0,-2,0,0,-2,0,0,0
14519,20000,2,13053,2,4078,16,2,0,0,-2,0,0,-2,0,0,-2,0,0,0


**Retards « payés » sous A qui restent en retard le mois suivant (règle 2)** (294 clients)

,LIMIT_BAL,PAY_1,BILL_AMT1,PAY_2,BILL_AMT2,PAY_AMT1,PAY_3,BILL_AMT3,PAY_AMT2,PAY_4,BILL_AMT4,PAY_AMT3,PAY_5,BILL_AMT5,PAY_AMT4,PAY_6,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,,,,,,,,,
3551,100000,3,2000,2,1000,0,-1,3000,3000,0,2000,0,0,1000,0,-1,1150,1150,0
1388,20000,1,14112,2,11221,3520,2,10656,0,0,7928,1000,-1,580,580,-1,0,0,0
16673,100000,-1,772,2,772,0,2,-8,8,-1,2562,2570,-1,0,0,-2,0,0,0
29120,200000,-1,2070,2,3250,2500,2,0,0,-1,1500,2250,0,750,0,-1,750,750,845
27312,280000,-1,652,3,652,0,2,326,0,-1,476,476,-1,2482,2482,-1,-101,101,326
14372,20000,0,20246,0,36350,3000,2,18910,3000,2,33500,290,2,-15306,5000,-1,20000,35306,0


**Comment lire** : une ligne par client ; pour le mois concerné, lire le trio codification, facture du mois, paiement du mois suivant.

## 6. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*